In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2 as cv
import seaborn as sns
import pandas as pd
import tensorflow as tf
from tensorflow.keras import preprocessing, layers, models, callbacks, optimizers
from sklearn import model_selection
from pathlib import Path
from tensorflow import keras
from numpy import random

In [2]:
sns.set()
sns.set_context("notebook")
plt.rcParams["figure.figsize"] = (5*16/9, 5)
plt.rcParams["figure.dpi"] = 100

# Data preprocessing

In [3]:
DATA_DIR = Path() / "data"
DATA_FOLDERS = [f for f in os.listdir(DATA_DIR) if "." not in f and f != "mixed"]

In [6]:
CATEGORY_MEMBERS = ['yeji', 'lia', 'ryujin', 'chaeryeong', 'yuna']
filenames, categories = [], []
for folder in DATA_FOLDERS:
    for file in os.listdir(f"{DATA_DIR}/{folder}"):
        filenames.append(f"{folder}/{file}")
        categories.append(folder)
        
df = pd.DataFrame(dict(filename=filenames, category=categories))
# df["category"] = df["category"].replace(dict(zip(CATEGORY_MEMBERS, range(len(CATEGORY_MEMBERS)))))
df

,filename,category
0,chaeryeong/726837221537087508.jpg,chaeryeong
1,chaeryeong/729954976100777985.jpg,chaeryeong
2,chaeryeong/729954977086308432.jpg,chaeryeong
3,chaeryeong/730000274843893770.jpg,chaeryeong
4,chaeryeong/730000275619709008.jpg,chaeryeong
...,...,...
2220,yuna/765378450478465024.jpg,yuna
2221,yuna/765378450759745546.jpg,yuna
2222,yuna/766541091917922314.jpg,yuna
2223,yuna/766541092446928906.jpg,yuna


In [22]:
train_df, test_df = model_selection.train_test_split(df, test_size=0.2, random_state=420)
train_df, val_df = model_selection.train_test_split(train_df, test_size=0.2, random_state=69)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

total_train = len(train_df)
total_val = len(val_df)
batch_size = 32
epochs = 50
img_w, img_h, img_ch = 224, 224, 3

In [23]:
len(train_df), len(val_df), len(test_df)

(1424, 356, 445)

In [24]:
train_datagen = preprocessing.image.ImageDataGenerator(
    rescale=1/255,
    rotation_range=15,
    shear_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1
)

train_gen = train_datagen.flow_from_dataframe(
    train_df,
    DATA_DIR,
    x_col="filename",
    y_col="category",
    target_size=(img_w, img_h),
    batch_size=batch_size,
    class_mode="categorical",
)

Found 1424 validated image filenames belonging to 5 classes.


In [25]:
val_datagen = preprocessing.image.ImageDataGenerator(rescale=1/255)

val_gen = val_datagen.flow_from_dataframe(
    val_df,
    DATA_DIR,
    x_col="filename",
    y_col="category",
    target_size=(img_w, img_h),
    batch_size=batch_size,
    class_mode="categorical",
)

Found 356 validated image filenames belonging to 5 classes.


In [26]:
def tfdata_generator(datagen):
    dataset = tf.data.Dataset.from_generator(
        lambda: datagen,
        output_types=(tf.float32, tf.float32),
        output_shapes=(tf.TensorShape([None, None, None, None]), tf.TensorShape([None, None])),
    )
    dataset = dataset.prefetch(tf.data.experimental.AUTOTUNE)
    return dataset

In [27]:
train_ds = tfdata_generator(train_gen)
val_ds = tfdata_generator(val_gen)

# Instantiate model

In [28]:
conv_layers = []
for i, f in enumerate([32, 64, 128, 128]):
    conv = layers.Conv2D(
        filters=f,
        kernel_size=3,
        activation="relu",
        name=f"conv{i+1}"
    )
    pool = layers.MaxPooling2D(
        pool_size=3,
        strides=2,
        name=f"pool{i+1}"
    )
    conv_layers.append(conv)
    conv_layers.append(pool)
    
model = models.Sequential(
    [
        layers.Input(shape=(img_w, img_h, img_ch), name="input"),
        *conv_layers,
        layers.Flatten(name="flatten"),
        layers.Dropout(0.5, name="drophalf"),
        layers.Dense(512, activation="relu"),
        layers.Dense(1, activation="relu"),
    ],
    name="kvision",
)

In [29]:
class StopOnValue(callbacks.Callback):
    def __init__(self, 
                 monitor='val_loss', 
                 value=0.00001, 
                 mode='min',
                 verbose=0):
        super(callbacks.Callback, self).__init__()
        self.monitor = monitor
        self.value = value
        self.verbose = verbose
        self.mode = mode
        if self.mode == 'min':
            self.compare_op = np.less
        elif self.mode == 'max':
            self.compare_op = np.greater
    
    def on_epoch_end(self, epoch, logs={}):
        current = logs.get(self.monitor)
        if current is None:
            warnings.warn('Early stopping requires %s available!' % self.monitor, RuntimeWarning)
            
        if self.compare_op(current, self.value):
            if self.verbose > 0:
                print('Epoch %05d: early stopping THR' % epoch)
            self.model.stop_training = True

In [30]:
stopval = StopOnValue(
    monitor="val_loss",
    value=0.1,
    mode="min",
    verbose=1,
)

lrred = callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=1/10,
    mode="min",
    min_lr=1e-4,
    verbose=1,
)

checkpoint = callbacks.ModelCheckpoint(
    "kvision.h5",
    monitor="val_loss",
    verbose=1,
    save_best_only=True,
    mode="min",
)

csv = callbacks.CSVLogger(
    "kvision.log",
    separator=",",
)

sgd = optimizers.SGD(
    lr=0.01,
    momentum=0.9,
    decay=5e-4,
)

In [31]:
model.compile(
    loss="categorical_crossentropy",
    optimizer=sgd,
    metrics=["acc"],
)

In [32]:
history = model.fit(
    train_ds,
    epochs=epochs,
    validation_data=val_ds,
    validation_steps=total_val//batch_size,
    steps_per_epoch=total_train//batch_size,
    callbacks=[lrred, checkpoint, csv],
    verbose=1,
)

Epoch 1/50
44/44 [==============================] - ETA: 0s - loss: 1.1921e-07 - acc: 0.8000
Epoch 00001: val_loss improved from inf to 0.00000, saving model to kvision.h5
44/44 [==============================] - 39s 890ms/step - loss: 1.1921e-07 - acc: 0.8000 - val_loss: 1.1921e-07 - val_acc: 0.8000
Epoch 2/50
44/44 [==============================] - ETA: 0s - loss: 1.1921e-07 - acc: 0.8000
Epoch 00002: val_loss did not improve from 0.00000
44/44 [==============================] - 36s 820ms/step - loss: 1.1921e-07 - acc: 0.8000 - val_loss: 1.1921e-07 - val_acc: 0.8000
Epoch 3/50
44/44 [==============================] - ETA: 0s - loss: 1.1921e-07 - acc: 0.8000
Epoch 00003: val_loss did not improve from 0.00000
44/44 [==============================] - 35s 794ms/step - loss: 1.1921e-07 - acc: 0.8000 - val_loss: 1.1921e-07 - val_acc: 0.8000
Epoch 4/50
44/44 [==============================] - ETA: 0s - loss: 1.1921e-07 - acc: 0.8000
Epoch 00004: val_loss did not improve from 0.00000
44/44 